# Tutorial for Training/Deploying the Physical CartPole Demo

This notebook follows the demo-hls4ml-25 README (Steps 1–7)

> Use the navigation links below to jump around quickly.


## Navigation
- [Step 0 — Notebook prerequisites](#step-0-notebook-prerequisites)
- [Step 1 — Environment Setup](#step-1-environment-setup)
- [Step 2 — Training Neural Network Controller](#step-2-training-neural-network-controller)
- [Step 3 — Running the Cartpole Simulator](#step-3-running-the-cartpole-simulator)
- [Step 4 — Convert Neural Network Controller with hls4ml](#step-4-convert-neural-network-controller-with-hls4ml)
- [Step 5 — Testing Model on PC (Local Hardware)](#step-5-testing-model-on-pc-local-hardware)
- [Step 6 — Implementation (Vivado/Vitis)](#step-6-implementation-vivadovitis)
- [Step 7 — Load Image on SD card and onto FPGA](#step-7)


## Step 0: Notebook prerequisites 

- Please make sure you have Conda installed
    - Refer to [this article](https://docs.conda.io/projects/conda/en/latest/user-guide/install/index.html) for instructions on setting it up
- Ensure you have access to these programs
    - Vivado 2020.1 
    - Vitis 2020.1


## Step 1: Environment Setup

### Conda environment
In your terminal run these
```bash
conda create -n physical_cartpole python=3.10
conda activate physical_cartpole
```
### Install Packages
These are just the necessary packages for training, we will install the GUI/Simulation packages later
<!-- GUI extras often needed in notebook environments:
## %pip install watchdog pydot graphviz PyQt6 -->
**Lab server note:** if your environment is pre-configured, you can skip this step.


In [ ]:
# Install base dependencies
%pip install -r ./Driver/CartPoleSimulation/CPS_list_of_packages.txt

# Notebook/GUI extras used in this walkthrough
%pip install seaborn watchdog pydot graphviz PyQt6

### 1.1 Choose experiment + model name & set paths for later

Edit these values directly in the next code cell:
- `EXPERIMENT_NAME = "Experiment-1"`
- `NET_NAME = "Dense-7IN-32H1-32H2-1OUT-0"`

If you want a different experiment/model, change those strings and rerun the setup cell.

Why paths are resolved here:
- Jupyter can start with different current working directories (repo root, a subfolder, or an external launch directory).
- Downstream steps call scripts with relative paths (`step3.sh`, SI_Toolkit configs, Vivado/Vitis scripts).
- We resolve `REPO` once in this setup cell and all later cells reuse it.


In [ ]:
import os, shutil
import sys

from pathlib import Path
from typing import Optional

def find_repo_root(start: Optional[Path] = None) -> Path:
    """Locate the physical-cartpole repo from varied notebook launch directories."""
    cursor = (start or Path.cwd()).resolve()
    for candidate in [cursor, *cursor.parents]:
        if (candidate / "Driver" / "CartPoleSimulation").exists() and (candidate / "README-demo-hls4ml-25.md").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate physical-cartpole repo root. "
        "Start Jupyter from this repository or one of its subdirectories."
    )

# Resolve once so all later cells use the same root path.
# Supported contexts:
# 1) Notebook launched from repo root
# 2) Notebook launched from a subdirectory in this repo
# 3) Jupyter launched elsewhere, then notebook opened from this repo
REPO = find_repo_root()
SIM = REPO / "Driver" / "CartPoleSimulation"
SI_ASF = SIM / "SI_Toolkit_ASF"
WORKSPACE = SI_ASF / "Experiments"
HLS_CONFIG = SI_ASF / "config_hls.yml"

# Set the experiment name here (edit this string for your run)
EXPERIMENT_NAME = "Experiment-1"

# Set the neural-network model name here (edit this string for your run)
NET_NAME = "Dense-7IN-32H1-32H2-1OUT-0"

# Path to seed (template) experiment that is provided
SEED_EXPERIMENT = REPO / "Experiment-1"

# Path to active experiment workspace used by SI_Toolkit
ACTIVE_EXPERIMENT = WORKSPACE / EXPERIMENT_NAME
MODELS_DIR = ACTIVE_EXPERIMENT / "Models"

# Ensure repo-local SI_Toolkit is importable without depending on editable installs.
si_src = SIM / "SI_Toolkit" / "src"
if str(si_src) not in sys.path:
    sys.path.insert(0, str(si_src))

# Ensure SI_Toolkit_ASF imports resolve from this repository checkout.
if str(SIM) not in sys.path:
    sys.path.insert(0, str(SIM))

# Echo resolved paths and selected model for verification
print("REPO:", REPO)
print("SIM:", SIM)
print("SI_ASF:", SI_ASF)
print("ACTIVE_EXPERIMENT:", ACTIVE_EXPERIMENT)
print("MODELS_DIR:", MODELS_DIR)
print("HLS_CONFIG:", HLS_CONFIG)
print("NET_NAME:", NET_NAME)

### 1.2 Move/copy seed experiment into the SI_Toolkit experiments folder

This shell scripts effectively ensure the experiment lives under `SI_Toolkit_ASF/Experiments/`.
We do a safe copy **only if missing**, to avoid overwriting trained artifacts.


In [ ]:
import shutil

if not ACTIVE_EXPERIMENT.exists():
    shutil.copytree(SEED_EXPERIMENT, ACTIVE_EXPERIMENT)
    print("Copied ./Experiment-1 →", ACTIVE_EXPERIMENT)
else:
    print("Active experiment already exists: not overwriting")

## Step 2: Training Neural Network Controller

### Dataset location
The Seed dataset is initially located at: `./Experiment-1` in the root directory which contains:
- recorded trajectories (CSV)
- a known good model configuration

Now there are **two paths**:

### Path A: Use precomputed model
- Use a pre-trained model from the seed folder `../Experiment-1/Models` (chosen in step 1)
- [Skip to the simulation (Step 3)](#step-3-running-the-cartpole-simulator)

### Path B: Train the neural network
- Follow the next steps

### 2.1 Training Configuration (config_training.yml)

Run this cell to see the current training configurations in:
`Driver/CartPoleSimulation/SI_Toolkit_ASF/config_training.yml`

Please refer to the [training_configurations_walkthrough](training_configurations_walkthrough.ipynb) notebook for more details on the training configurations

In [ ]:
import yaml
from pprint import pprint

training_config_path = SI_ASF / "config_training.yml"

with training_config_path.open("r") as f:
    cfg = yaml.safe_load(f)

print("Loaded:", training_config_path)
pprint(cfg)


Below is an **optional** cell that updates the yaml's experiment path if you changed it

In [ ]:
import yaml

cfg_path = SI_ASF / "config_training.yml"
cfg = yaml.safe_load(cfg_path.read_text())

def set_experiment_path(cfg_obj, new_path: str) -> bool:
    if isinstance(cfg_obj, dict):
        if "paths" in cfg_obj and isinstance(cfg_obj["paths"], dict) and "path_to_experiment" in cfg_obj["paths"]:
            cfg_obj["paths"]["path_to_experiment"] = new_path
            return True
        if "PATH_TO_EXPERIMENT" in cfg_obj:
            cfg_obj["PATH_TO_EXPERIMENT"] = new_path
            return True
    return False

ok = set_experiment_path(cfg, str(ACTIVE_EXPERIMENT))
if ok:
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("Updated config_training.yml to use:", ACTIVE_EXPERIMENT)
else:
    print("Could not locate experiment path key; set it manually to:", ACTIVE_EXPERIMENT)

### 2.2 Data normalization

Neural networks train much more reliably when each feature is on a comparable numeric scale.

In this project, **normalization statistics are computed from the `Train/` CSVs** inside the active experiment folder.  
Those statistics are saved to `NormalizationInfo/` and then reused consistently for:

- training 
- simulation inference
- FPGA/HLS deployment

The function that does this is `SI_Toolkit.load_and_normalize.calculate_normalization_info(...)`.


In [ ]:
# Load the training configuration
import yaml

cfg_path = SI_ASF / "config_training.yml"
cfg = yaml.safe_load(cfg_path.read_text())

# The two path keys
print("PATH_TO_EXPERIMENT_FOLDERS =", cfg["paths"]["PATH_TO_EXPERIMENT_FOLDERS"])
print("path_to_experiment         =", cfg["paths"]["path_to_experiment"])
print("DATA_FOLDER                =", cfg["paths"]["DATA_FOLDER"])

train_dir = Path(cfg["paths"]["PATH_TO_EXPERIMENT_FOLDERS"]) / cfg["paths"]["path_to_experiment"] / cfg["paths"]["DATA_FOLDER"] / "Train"
print("\nTrain directory (resolved):", train_dir.resolve())


In [ ]:
# Identify the CSVs that are used for normalization
import glob

train_csvs = sorted(glob.glob(str(train_dir / "*.csv")))
print(f"Found {len(train_csvs)} training CSV files.")
for p in train_csvs[:10]:
    print(" -", p)
if len(train_csvs) > 10:
    print(f" ... ({len(train_csvs)-10} more)")


In [ ]:
from SI_Toolkit.load_and_normalize import get_paths_to_datafiles, load_data

# Build list of CSV paths
train_paths = get_paths_to_datafiles(str(train_dir))
val_dir = train_dir.parent / "Validation"
test_dir = train_dir.parent / "Test"

# Load CSVs into DataFrames
training_dfs = load_data(train_paths)

validation_dfs = load_data(get_paths_to_datafiles(str(val_dir)))
test_dfs       = load_data(get_paths_to_datafiles(str(test_dir)))

print("Loaded training files:", len(training_dfs))
print("Example columns:", training_dfs[0].columns.tolist())

#### 2.2.1 Compute normalization statistics + write `NormalizationInfo/NI_*.csv`

By default the function:
- concatenates all Train CSVs into one DataFrame
- drops a `time` column (if present)
- computes per-feature **mean/std/min/max**
- optionally applies a **user correction hook** from `SI_Toolkit_ASF/ToolkitCustomization/...`
- writes a timestamped `NI_YYYY-MM-DD_HH-MM-SS.csv`
- optionally saves histogram PNGs per feature


In [ ]:
# Run the normalization (preprocessing) step
from SI_Toolkit.load_and_normalize import calculate_normalization_info

# Keep histograms ON for now, optionally turn OFF for speed in automated runs
df_norm_info, norm_csv_path = calculate_normalization_info(
    config=cfg,
    plot_histograms=True,
    user_correction=False, 
)

print("Wrote normalization CSV:", norm_csv_path)
display(df_norm_info)

#### 2.2.2 How to read `df_norm_info`

`df_norm_info` is indexed by the statistic (`mean`, `std`, `min`, `max`).  
Each column is a feature from your training CSVs (excluding `time`).

Typical z-score normalization uses:

- **normalize**:  \(x_{norm} = (x - \mu) / \sigma\)
- **denormalize**: \(x = x_{norm} \cdot \sigma + \mu\)

The exact columns used as inputs/targets are determined by your training config and the model wrapper,
but the statistics come from the raw CSV columns.


#### 2.2.3 Where the histograms went

If `plot_histograms=True`, the function saves one histogram per feature to:

`.../NormalizationInfo/histograms/<feature>.png`

In [ ]:
# Show a couple histogram images
from pathlib import Path

hist_dir = Path(norm_csv_path).parent / "histograms"
print("Histogram dir:", hist_dir)

if hist_dir.exists():
    pngs = sorted(hist_dir.glob("*.png"))
    print("Found", len(pngs), "histograms.")
else:
    print("No histograms folder found (plot_histograms may be False).")


### 2.3 Train the neural network
At this point, we have written a normalization file from the **Train** split. Now we go through the training flow.

Training is config driven, uses the normalization statistics computed above, and writes a fully reproducible model folder under `Models/`.

#### 2.3.1 Load training arguments (YAML + CLI overrides)

Training is config driven: `args()` loads `config_training.yml` defaults and applies any CLI overrides.
We print the resolved paths and split files so you can confirm the **active experiment**.


In [ ]:
## Set the correct path to run training 
import os
from pathlib import Path

if "SIM" not in globals():
    raise RuntimeError("Run Step 1.1 first so shared paths (REPO/SIM) are defined.")

print("Setting working directory to:", SIM)
os.chdir(SIM)

print("Now cwd =", Path.cwd())
print("Config exists? ", (SIM / "SI_Toolkit_ASF" / "config_training.yml").exists())


In [ ]:
from SI_Toolkit.Functions.General.load_parameters_for_training import args 
from SI_Toolkit.Functions.General.Initialization import set_seed

# Save Jupyter argv 
_argv_backup = sys.argv.copy()

# Make argparse think we're running with no CLI args
sys.argv = [sys.argv[0]]

# This parses the yaml file for our training details. 
# here we can change things like batch size and epochs
a = args()

# Sets random seed
set_seed(a)

# Restore argv
sys.argv = _argv_backup
print("Key training settings:")
for k in ["net_name", "library", "path_to_models", "training_files", "validation_files", "test_files", "config_path"]:
    if hasattr(a, k):
        print(f"  {k}: {getattr(a, k)}")


#### 2.3.2 Instantiate the model and inspect data

`get_net(a)` builds the network and a `net_info` object that describes naming, paths, and whether normalization is enabled. `create_full_name(...)` generates the run folder name.


In [ ]:
from SI_Toolkit.Functions.General.Initialization import get_net, create_full_name

net, net_info = get_net(a)
create_full_name(net_info, a.path_to_models)

print("Model full name:", net_info.net_full_name)
print("Backend library:", net_info.library)
print("Will normalize:", getattr(net_info, "normalize", None))
print("Models root folder:", a.path_to_models)
print("This run will be saved under:", f"{a.path_to_models}/{net_info.net_full_name}")


#### 2.3.3 Load normalization info

This is why normalization must run first: training needs the mean/std (and related vectors) to scale features.
These vectors are also reused later for consistent inference (including HLS/firmware).


In [ ]:
from SI_Toolkit.Functions.General.Initialization import get_norm_info_for_net
from SI_Toolkit.Functions.General.Normalising import write_out_normalization_vectors

normalization_info = get_norm_info_for_net(net_info, files_for_normalization=a.training_files)
write_out_normalization_vectors(normalization_info, net_info)

print("Normalization info loaded and vectors written for:", net_info.net_full_name)

#### 2.3.4 Load Train/Validate/Test CSVs 

Here we resolve file lists for each split, load them as DataFrames, and print basic counts. This connects CSV logs to the training tensors.


In [ ]:
from SI_Toolkit.load_and_normalize import load_data, get_paths_to_datafiles
from SI_Toolkit.Functions.General.Initialization import create_log_file
import os

train_paths = get_paths_to_datafiles(a.training_files)
val_paths   = get_paths_to_datafiles(a.validation_files)
test_paths  = get_paths_to_datafiles(a.test_files)

training_dfs   = load_data(train_paths)
validation_dfs = load_data(val_paths)
test_dfs       = load_data(test_paths)

run_dir = os.path.join(a.path_to_models, net_info.net_full_name)
os.makedirs(run_dir, exist_ok=True)

create_log_file(net_info, a, training_dfs)

def nrows(dfs):
    return sum(len(df) for df in dfs) if isinstance(dfs, list) else len(dfs)

print("Train files:", len(train_paths), "rows:", nrows(training_dfs))
print("Val files:  ", len(val_paths),   "rows:", nrows(validation_dfs))
print("Test files: ", len(test_paths),  "rows:", nrows(test_dfs))

example_cols = training_dfs[0].columns if isinstance(training_dfs, list) else training_dfs.columns
print("Example columns:", list(example_cols))


#### 2.3.5 Add Previous Control Input (`Q_applied_-1`) and Repair Normalization

This step augments each dataset with a one timestep delayed control input.
For every DataFrame, we sort by time, shift `Q_applied` by one step to create `Q_applied_-1`, and drop the first row (which has no previous value).

Because `Q_applied_-1` is derived after loading CSVs, older normalization files may have missing/NaN stats for this column.
The next code cell automatically repairs normalization statistics for any required input/output column that is missing or nonfinite.


In [ ]:
import numpy as np
import pandas as pd

def add_prev_Q_applied(dfs, time_col="time"):
    for df in dfs:
        if "Q_applied_-1" in df.columns:
            continue

        # Ensure sorted by time just in case
        if time_col in df.columns:
            df.sort_values(time_col, inplace=True)

        df["Q_applied_-1"] = df["Q_applied"].shift(1)

        # First row has no previous value -> drop it
        df.dropna(subset=["Q_applied_-1"], inplace=True)
        df.reset_index(drop=True, inplace=True)

def ensure_normalization_stats(normalization_info, dfs_for_stats, required_columns):
    repaired = []

    for col in required_columns:
        missing_col = col not in normalization_info.columns
        invalid_col = (not missing_col) and normalization_info[col].isna().any()

        if not (missing_col or invalid_col):
            continue

        values = []
        for df in dfs_for_stats:
            if col in df.columns:
                s = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64, copy=False)
                s = s[np.isfinite(s)]
                if s.size > 0:
                    values.append(s)

        if not values:
            raise ValueError(
                f"Cannot repair normalization for '{col}': column missing/non-finite in all provided DataFrames."
            )

        x = np.concatenate(values)
        normalization_info.loc["mean", col] = float(np.mean(x))
        normalization_info.loc["std", col] = float(np.std(x))
        normalization_info.loc["max", col] = float(np.max(x))
        normalization_info.loc["min", col] = float(np.min(x))
        repaired.append(col)

    return repaired

add_prev_Q_applied(training_dfs)
add_prev_Q_applied(validation_dfs)
add_prev_Q_applied(test_dfs)

required_cols = sorted(set(net_info.inputs) | set(net_info.outputs))
repaired_cols = ensure_normalization_stats(
    normalization_info,
    training_dfs + validation_dfs,
    required_cols,
)

if repaired_cols:
    print("Repaired normalization stats for:", repaired_cols)
else:
    print("Normalization stats already valid for all required columns")

# Verify one training dataframe
df0 = training_dfs[0]
print(df0[["time", "Q_applied", "Q_applied_-1", "Q_calculated"]].head(5))

#### 2.3.6 Run the training loop

This is the real training loop used by `train_network()`: it calls `Training.train_network_core(...)`.
It returns loss curves used to generate the training plot.

A preflight data check runs in the previous cell and blocks training if non-finite values are detected.

**Note:** Training can take a long time.


In [ ]:
if net_info.library == "TF":
    import SI_Toolkit.Functions.TF.Training as Training
else:
    import SI_Toolkit.Functions.Pytorch.Training as Training

from SI_Toolkit.Functions.General.TerminalContentManager import TerminalContentManager

with TerminalContentManager(os.path.join(run_dir, "terminal_output.txt")):
    loss, val_loss, post_epoch_loss = Training.train_network_core(
        net, net_info,
        training_dfs,
        validation_dfs,
        test_dfs,
        normalization_info,
        a
    )

print("Final validation loss:", val_loss[-1])

#### 2.5.6 Plot losses and point to saved artifacts

After training finishes, the run folder under `Models/` contains:
- the trained model/checkpoints
- `terminal_output.txt` (captured outputs)
- copied config + training script
- `training_curve.png`


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plt.figure()
plt.plot(loss, label="train")
plt.plot(val_loss, label="val")
plt.yscale("log")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title(net_info.net_full_name)
plt.savefig(os.path.join(net_info.path_to_net, "training_curve.png"))
plt.show()

model_dir = Path(a.path_to_models) / net_info.net_full_name
print("Model artifacts saved in:", model_dir)
print("Look for: training_curve.png, terminal_output.txt, config copy, checkpoints/models")

NET_NAME = net_info.net_full_name
print("NET_NAME updated to trained run:", NET_NAME)


## Step 3: Running the Cartpole Simulator
In this section, we will primarily focus on using the Cartpole Simulator to test the performance of trained neural network controllers. This simulator not only allows for performance evaluation but can also be used to generate new datasets for further training. Here, we will describe how to effectively use the simulator to assess your model's capabilities.

### 3.1 Choose the Model
   - Run the program with the desired model name as an argument to select a specific trained model. If no model name is provided, the default pre-trained model will be used.
   - Available model names can be found in the following folder:
     ```
     Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1/Models
     ```

### 3.2 Run the GUI
- Make sure you have access to a display variable


In [ ]:
import subprocess
from pathlib import Path

if "REPO" not in globals():
    raise RuntimeError("Run Step 1.1 first so REPO is defined.")

script = REPO / "step3.sh"
if not script.exists():
    raise FileNotFoundError(f"Missing simulator launcher: {script}")
script.chmod(script.stat().st_mode | 0o111)

args = ["bash", str(script)]
if NET_NAME:
    args.append(NET_NAME)

print("Notebook CWD:", Path.cwd().resolve())
print("Repo root:", REPO)
print("Running:", " ".join(args))

# Run from repo root so relative paths inside step3.sh resolve.
subprocess.run(args, cwd=str(REPO), check=True)


## Step 4: Convert Neural Network Controller with hls4ml

This section converts the selected/trained neural controller into HLS using the SI_Toolkit + hls4ml pipeline.


### 4.1 Select the model to convert

If Step 2 training was run in this notebook, we use that exact run name (`net_info.net_full_name`).
Otherwise we fall back to `NET_NAME`.


In [ ]:
import os
import shutil
import subprocess
import yaml
from datetime import datetime
from pathlib import Path

# Prefer the trained model from this notebook run when available
if "net_info" in globals() and hasattr(net_info, "net_full_name"):
    SELECTED_NET_NAME = net_info.net_full_name
else:
    SELECTED_NET_NAME = NET_NAME

# Prefer Experiment-1 models dir if present, otherwise keep active experiment path
if (SI_ASF / "Experiments" / "Experiment-1" / "Models").exists():
    HLS_MODELS_DIR = SI_ASF / "Experiments" / "Experiment-1" / "Models"
else:
    HLS_MODELS_DIR = MODELS_DIR

print("SELECTED_NET_NAME:", SELECTED_NET_NAME)
print("HLS_MODELS_DIR:", HLS_MODELS_DIR)
print("Model folder exists:", (HLS_MODELS_DIR / SELECTED_NET_NAME).exists())


### 4.2 `config_hls.yml` in detail

Use the [hls4ml_config_hls_walkthrough](hls4ml_config_hls_walkthrough.ipynb) notebook to learn more about these parameters.

That notebook explains each key and how it impacts synthesis/resource/latency behavior.

#### 4.2.1 Load and inspect current `config_hls.yml`


In [ ]:
with HLS_CONFIG.open("r") as f:
    hls_cfg = yaml.safe_load(f)

print("Loaded:", HLS_CONFIG)
for k in [
    "path_to_hls_installation",
    "path_to_models",
    "net_name",
    "batch_size",
    "Strategy",
    "ReuseFactor",
    "backend",
    "output_dir",
]:
    print(f"{k}: {hls_cfg.get(k)}")

print("PRECISION:")
for k, v in hls_cfg.get("PRECISION", {}).items():
    print(f"  {k}: {v}")


#### 4.2.2 How each configuration family affects synthesis

- `PRECISION`: There are fixed point width/integer choices, larger widths usually improve accuracy, but increase resource usage.
- `Strategy`: `Resources` tends to trade latency for lower area; `Latency` pushes more parallelism.
- `ReuseFactor`: Higher reuse typically lowers DSP usage and increases latency.
- `backend`/`part`: Select synthesis backend and target FPGA part.
- `path_to_models` + `net_name`: Choose which trained model checkpoint folder is converted.
- `output_dir`: Controls where generated project and reports are emitted.


#### 4.2.3 Resolve run specific paths (paths, model, output folder)


In [ ]:
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")

# Vivado path: prefer env var XILINX_VIVADO, fallback to existing config value
vivado_root = os.environ.get("XILINX_VIVADO", "").strip()
if vivado_root:
    vivado_bin = Path(vivado_root) / "bin"
else:
    vivado_bin = Path(str(hls_cfg.get("path_to_hls_installation", "")))

if not vivado_bin.exists():
    raise FileNotFoundError(
        f"Vivado bin path not found: {vivado_bin}. Set XILINX_VIVADO to your Vivado 2020.1 installation root."
    )

if not (HLS_MODELS_DIR / SELECTED_NET_NAME).exists():
    raise FileNotFoundError(f"Selected model directory not found: {HLS_MODELS_DIR / SELECTED_NET_NAME}")

HLS_OUTPUT_NAME = os.environ.get("CARTPOLE_HLS_OUTPUT", f"{SELECTED_NET_NAME}_{RUN_TAG}")
HLS_OUTPUT_ABS = REPO / "HLS4ML" / HLS_OUTPUT_NAME
HLS_OUTPUT_REL = os.path.relpath(HLS_OUTPUT_ABS, SIM)

print("Resolved Vivado bin:", vivado_bin)
print("Resolved model dir:", HLS_MODELS_DIR / SELECTED_NET_NAME)
print("Resolved output dir:", HLS_OUTPUT_ABS)
print("Resolved output rel:", HLS_OUTPUT_REL)


#### 4.2.4 Backup and write `config_hls.yml`


In [ ]:
backup_path = HLS_CONFIG.with_suffix(f".yml.bak_{RUN_TAG}")
shutil.copy2(HLS_CONFIG, backup_path)

hls_cfg["path_to_hls_installation"] = str(vivado_bin)
hls_cfg["path_to_models"] = os.path.relpath(HLS_MODELS_DIR, SIM)
hls_cfg["net_name"] = SELECTED_NET_NAME
hls_cfg["output_dir"] = HLS_OUTPUT_REL

with HLS_CONFIG.open("w") as f:
    yaml.safe_dump(hls_cfg, f, sort_keys=False)

print("Backup created:", backup_path)
print("Updated config:", HLS_CONFIG)


#### 4.3 Go through the conversion pipeline

1. load `config_hls.yml`
2. load model via `get_net(...)`
3. copy model artifacts to `output_dir`
4. run `convert_model_with_hls4ml(...)`
5. `compile`/`build`/Vivado report through hls4ml


In [ ]:
import hls4ml
import os
import shutil
import yaml
from pathlib import Path
from types import SimpleNamespace

from SI_Toolkit.Functions.General.Initialization import get_net
from SI_Toolkit.Functions.General.TerminalContentManager import TerminalContentManager

os.environ["OSTYPE"] = "linux-gnu"

cfg_path = SIM / "SI_Toolkit_ASF" / "config_hls.yml"
with cfg_path.open("r") as f:
    cfg_local = yaml.safe_load(f)

a_local = SimpleNamespace()
a_local.path_to_models = cfg_local["path_to_models"]
a_local.net_name = cfg_local["net_name"]
batch_size = cfg_local["batch_size"]

print("Config path:", cfg_path)
print("Using model:", a_local.net_name)
print("Using model root:", a_local.path_to_models)
print("Output dir:", cfg_local["output_dir"])

old_cwd = Path.cwd()
os.chdir(SIM)
try:
    from SI_Toolkit.HLS4ML.hls4ml_functions import convert_model_with_hls4ml

    model_dir = Path(a_local.path_to_models) / a_local.net_name
    if not model_dir.exists():
        raise FileNotFoundError(f"Model directory does not exist: {model_dir}")

    ckpt_candidates = [
        model_dir / f"{a_local.net_name}.ckpt",
        model_dir / f"{a_local.net_name}.ckpt.index",
        model_dir / "ckpt.ckpt",
        model_dir / "ckpt.ckpt.index",
        model_dir / f"{a_local.net_name}.pt",
        model_dir / "ckpt.pt",
    ]
    with TerminalContentManager(os.path.join(cfg_local["output_dir"], "terminal_output.txt")):
        net_local, net_info_local = get_net(
            a_local,
            time_series_length=1,
            batch_size=batch_size,
            stateful=True,
            remove_redundant_dimensions=True,
        )

        path_to_network = os.path.join(a_local.path_to_models, a_local.net_name)
        path_to_hls_network = os.path.join(cfg_local["output_dir"], a_local.net_name)
        shutil.copytree(path_to_network, path_to_hls_network, dirs_exist_ok=True)

        hls_model_local, hls_model_cfg = convert_model_with_hls4ml(net_local)
        hls4ml.utils.plot_model(hls_model_local, show_shapes=True, show_precision=True, to_file=None)

        hls_model_local.build(
            reset=False,      # Reuse existing HLS project directory if it already exists.
            csim=True,        # Run C simulation (functional check before RTL generation).
            synth=True,       # Run HLS synthesis (C/C++ -> RTL with resource/latency estimates).
            cosim=True,       # Run C/RTL co-simulation to compare C model vs generated RTL.
            validation=True,  # Run hls4ml validation checks on generated outputs.
            export=True,      # Package/export generated IP/project artifacts to be used with Vivado.
            vsynth=True,      # Run downstream Vivado synthesis on exported design.
        )
        hls4ml.report.read_vivado_report(cfg_local["output_dir"])
finally:
    os.chdir(old_cwd)

print("Conversion completed")

### 4.4 Verify generated artifacts

Primary output location:
`HLS4ML/<run>/myproject_prj/solution1/impl/vhdl`


In [ ]:
vhdl_dir = HLS_OUTPUT_ABS / "myproject_prj" / "solution1" / "impl" / "vhdl"
terminal_log = HLS_OUTPUT_ABS / "terminal_output.txt"

print("HLS output dir:", HLS_OUTPUT_ABS)
print("terminal_output.txt exists:", terminal_log.exists())
print("VHDL dir exists:", vhdl_dir.exists())

if vhdl_dir.exists():
    vhdl_files = sorted(vhdl_dir.glob("*.vhd"))
    print("VHDL file count:", len(vhdl_files))
    for pth in vhdl_files[:12]:
        print(" -", pth.name)

reports = sorted(HLS_OUTPUT_ABS.glob("**/*.rpt"))
print("Report count:", len(reports))
for pth in reports[:12]:
    print(" -", pth.relative_to(HLS_OUTPUT_ABS))


## Step 5: Testing Model on PC (Local Hardware)

This step validates your trained model using the real cartpole from a local machine.
You will set:
- active controller selection
- serial-port behavior (optional manual override)
- neural-imitator model config

Calibration values from this step are reused in Step 6.

Please also review the calibration guide in [`README.md` (Calibration)](README.md#calibration).


### 5.0 Local Execution Requirement

Run this step on your own machine with the cartpole connected over USB. Do not run it from the lab server.

If needed, copy the repository from a lab server:

```bash
scp -r asuad\yourasuID@129.219.30.13:~/project/cartpole/physical-cartpole ~/Desktop/physical-cartpole
```


### 5.1 Set Controller to `neural-imitator`

Target file: `Driver/globals.py`

This helper cell updates `CONTROLLER_NAME` to `neural-imitator`.

In [ ]:
from pathlib import Path
import re

GLOBALS_PY = REPO / "Driver" / "globals.py"
APPLY_CONTROLLER_PATCH = True

text = GLOBALS_PY.read_text()
match = re.search(r'^CONTROLLER_NAME\s*=\s*[\'"]([^\'"]+)[\'"]', text, flags=re.M)
print("globals.py:", GLOBALS_PY)
print("Current CONTROLLER_NAME:", match.group(1) if match else "not found")

new_text, count = re.subn(
    r'^CONTROLLER_NAME\s*=\s*[\'"][^\'"]+[\'"]',
    "CONTROLLER_NAME = 'neural-imitator'",
    text,
    count=1,
    flags=re.M,
)
if count != 1:
    raise RuntimeError("Could not patch CONTROLLER_NAME in globals.py")

if APPLY_CONTROLLER_PATCH:
    GLOBALS_PY.write_text(new_text)
    print("Patched CONTROLLER_NAME -> neural-imitator")
else:
    print("Dry run only. Set APPLY_CONTROLLER_PATCH=True to write.")


### 5.2 Serial Port Configuration (Optional Manual Override)

Target file: `Driver/DriverFunctions/interface.py`

If auto-detection fails (common on some macOS setups), this helper can add/use a manual serial-port override.

1. Set `SERIAL_PORT_OVERRIDE` to your device path.

Tip: on macOS, list ports with `ls /dev/tty.*` while the board is connected and powered.


In [ ]:
from pathlib import Path
import re

INTERFACE_PY = REPO / "Driver" / "DriverFunctions" / "interface.py"
SERIAL_PORT_OVERRIDE = "/dev/tty.usbserial-210351B7BD461"  # Replace with your local port path
APPLY_SERIAL_PATCH = True

text = INTERFACE_PY.read_text()
print("interface.py:", INTERFACE_PY)

if "MANUAL_SERIAL_PORT =" not in text:
    text = text.replace(
        "import pandas as pd\n",
        "import pandas as pd\n\nMANUAL_SERIAL_PORT = None  # e.g. '/dev/tty.usbserial-XXXX'\n",
        1,
    )

if "Using manual serial port override" not in text:
    marker = "    from serial.tools import list_ports\n"
    text = text.replace(
        marker,
        "    if MANUAL_SERIAL_PORT:\n"
        "        print(f'Using manual serial port override: {MANUAL_SERIAL_PORT}')\n"
        "        return MANUAL_SERIAL_PORT\n\n" + marker,
        1,
    )

text, n = re.subn(
    r"^MANUAL_SERIAL_PORT\s*=.*$",
    f"MANUAL_SERIAL_PORT = {SERIAL_PORT_OVERRIDE!r}",
    text,
    count=1,
    flags=re.M,
)
if n != 1:
    raise RuntimeError("Could not set MANUAL_SERIAL_PORT")

if APPLY_SERIAL_PATCH:
    INTERFACE_PY.write_text(text)
    print("Patched manual serial-port override")
else:
    print("Dry run only. Set APPLY_SERIAL_PATCH=True to write.")


### 5.3 Model Configuration for `neural-imitator`

Target file: `Driver/CartPoleSimulation/Control_Toolkit_ASF/config_controllers.yml`

This helper updates key fields under `neural-imitator`:
- `PATH_TO_MODELS`
- `net_name`
- `input_precision`
- `hls4ml`

In [ ]:
from pathlib import Path
import yaml

CONTROLLERS_YML = REPO / "Driver" / "CartPoleSimulation" / "Control_Toolkit_ASF" / "config_controllers.yml"
APPLY_MODEL_CONFIG_PATCH = True

net_name = globals().get("SELECTED_NET_NAME", "Dense-7IN-32H1-32H2-1OUT-0")
path_to_models = "./CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1/Models/"

cfg = yaml.safe_load(CONTROLLERS_YML.read_text())
ni = cfg.setdefault("neural-imitator", {})

print("config_controllers.yml:", CONTROLLERS_YML)
print("Current PATH_TO_MODELS:", ni.get("PATH_TO_MODELS"))
print("Current net_name:", ni.get("net_name"))
print("Current input_precision:", ni.get("input_precision"))
print("Current hls4ml:", ni.get("hls4ml"))

ni["PATH_TO_MODELS"] = path_to_models
ni["net_name"] = net_name
ni["input_precision"] = "float"
ni["hls4ml"] = False

if APPLY_MODEL_CONFIG_PATCH:
    CONTROLLERS_YML.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("Patched neural-imitator model configuration")
else:
    print("Dry run only. Set APPLY_MODEL_CONFIG_PATCH=True to write.")


### 5.4 Run PC Control Software

From the repository root (`physical-cartpole/`):

```bash
export PYTHONPATH=$(pwd):$PYTHONPATH
python Driver/control.py
```

Useful keys in the running app:
- `h`: help
- `K`: calibrate track middle
- `k`: PC control on/off
- `u`: chip control on/off
- `D`: dance mode on/off

Calibration outputs from this step (motor power, track middle behavior, vertical angle) are required for Step 6 firmware/SoC setup.


In [ ]:
import os

os.environ["PYTHONPATH"] = f"{REPO}:{os.environ.get('PYTHONPATH', '')}"
print("PYTHONPATH updated for this kernel.")
print("Run locally from repo root:")
print("  export PYTHONPATH=$(pwd):$PYTHONPATH")
print("  python Driver/control.py")


### 5.5 Calibration Notes

During this step, verify:
1. Middle of track calibration (`K`) each power cycle.
2. Motor power is sufficient to avoid sticking near boundaries.
3. Vertical angle/potentiometer settings are correct.

`ANGLE_HANGING_POLOLU` and `MOTOR_CORRECTION` values obtained here are reused in Step 6.


## Step 6: Implementation (Vivado/Vitis)

1. Generate the NN model bitstream (Vivado)
2. Generate the Zynq SoC project and `BOOT.bin` (Vitis/XSCT)


### 6.1 Preparation

Before you run synthesis/build cells, verify toolchain availability and patch firmware source values with your calibration/model outputs.


In [ ]:
import os
import re
import shutil
import subprocess
from pathlib import Path

if "REPO" not in globals():
    raise RuntimeError("Run Step 1.1 first so REPO is defined once for this notebook.")

IMPL_REPO = REPO
print("Implementation repo root:", IMPL_REPO)

required = [
    IMPL_REPO / "install_zybo_board.sh",
    IMPL_REPO / "generate_bitstream.tcl",
    IMPL_REPO / "generate_vitis_project.tcl",
    IMPL_REPO / "Firmware" / "create_symlinks_cartpole.sh",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required implementation files:\n - " + "\n - ".join(missing))

print("vivado:", shutil.which("vivado") or "NOT FOUND in PATH")
print("xsct:", shutil.which("xsct") or "NOT FOUND in PATH")
print("Vivado version target: 2020.1")
print("Vitis version target: 2020.1")


In [ ]:
symlink_script = IMPL_REPO / "Firmware" / "create_symlinks_cartpole.sh"
text = symlink_script.read_text()
lines = text.splitlines()

header_idx = None
for i, line in enumerate(lines):
    if line.strip().startswith("# For NeuralImitator on Zynq"):
        header_idx = i
        break

if header_idx is None:
    print("Warning: Could not find NeuralImitator block in create_symlinks_cartpole.sh")
else:
    next_line = lines[header_idx + 1].strip() if header_idx + 1 < len(lines) else ""
    if next_line == ":'":
        print("NeuralImitator symlink block already disabled (commented).")
    else:
        close_idx = None
        for j in range(header_idx + 1, len(lines)):
            if lines[j].strip() == ")":
                close_idx = j
                break
        if close_idx is None:
            raise RuntimeError("Could not find end of NeuralImitator array block for patching")
        lines.insert(header_idx + 1, ":'")
        lines.insert(close_idx + 2, "'")
        symlink_script.write_text("\n".join(lines) + "\n")
        print("Disabled NeuralImitator block in create_symlinks_cartpole.sh")

# Make automation scripts executable
for rel in ["install_zybo_board.sh", "generate_bitstream.tcl", "generate_vitis_project.tcl"]:
    path = IMPL_REPO / rel
    path.chmod(path.stat().st_mode | 0o111)
    print(f"Executable bit set: {path}")


In [ ]:
# Optional: patch Step 5 calibration values into firmware parameters.c
import re

PARAMETERS_C = IMPL_REPO / "Firmware" / "Src" / "CartPoleFirmware" / "parameters.c"
APPLY_PARAMETER_PATCH = True
NEW_MOTOR_CORRECTION = [0.6310468, 0.0472680, 0.0408973]
NEW_ANGLE_HANGING_POLOLU = 783.0

text = PARAMETERS_C.read_text()
block = re.search(r"#elif defined\(ZYNQ\)(.*?)#endif", text, flags=re.S)
if not block:
    raise RuntimeError("Could not find ZYNQ block in parameters.c")

curr = block.group(1)
print("Current MOTOR_CORRECTION:", re.search(r"MOTOR_CORRECTION\[3\]\s*=\s*\{([^}]*)\};", curr).group(1).strip())
print("Current ANGLE_HANGING_POLOLU:", re.search(r"ANGLE_HANGING_POLOLU\s*=\s*([0-9eE+\-.]+);", curr).group(1).strip())

patched = re.sub(r"float\s+MOTOR_CORRECTION\[3\]\s*=\s*\{[^}]*\};", f"float MOTOR_CORRECTION[3] = {{{', '.join(f'{v:.7f}' for v in NEW_MOTOR_CORRECTION)}}};", curr, count=1)
patched = re.sub(r"float\s+ANGLE_HANGING_POLOLU\s*=\s*[0-9eE+\-.]+;", f"float ANGLE_HANGING_POLOLU = {float(NEW_ANGLE_HANGING_POLOLU):.4f};", patched, count=1)

if APPLY_PARAMETER_PATCH:
    text = text[:block.start(1)] + patched + text[block.end(1):]
    PARAMETERS_C.write_text(text)
    print("Updated parameters.c")
else:
    print("Dry run only. Set APPLY_PARAMETER_PATCH=True to write.")


In [ ]:
# Optional: patch normalization vectors + header constants for neural-imitator firmware
from pathlib import Path
import csv
import re

APPLY_NORMALIZATION_PATCH = True
UPDATE_NETWORK_HEADER = True

# Resolve model directory
if "HLS_MODELS_DIR" in globals() and "SELECTED_NET_NAME" in globals():
    MODEL_DIR = Path(HLS_MODELS_DIR) / str(SELECTED_NET_NAME)
else:
    MODEL_DIR = REPO / "Driver" / "CartPoleSimulation" / "SI_Toolkit_ASF" / "Experiments" / "Experiment-1" / "Models" / str(globals().get("NET_NAME", "Dense-7IN-32H1-32H2-1OUT-0"))

# Resolve firmware files (support both hyphen and underscore names)
def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

NEURAL_C = first_existing([
    IMPL_REPO / "Firmware" / "Src" / "Zynq" / "neural_imitator.c",
    IMPL_REPO / "Firmware" / "Src" / "Zynq" / "neural-imitator.c",
])
NEURAL_H = first_existing([
    IMPL_REPO / "Firmware" / "Src" / "Zynq" / "neural_imitator.h",
    IMPL_REPO / "Firmware" / "Src" / "Zynq" / "neural-imitator.h",
])
if NEURAL_C is None or NEURAL_H is None:
    raise FileNotFoundError("Could not find neural_imitator.c/.h under Firmware/Src/Zynq")

files = {
    "hls_normalize_a": MODEL_DIR / "normalization_vec_a.csv",
    "hls_normalize_b": MODEL_DIR / "normalization_vec_b.csv",
    "hls_denormalize_A": MODEL_DIR / "denormalization_vec_A.csv",
    "hls_denormalize_B": MODEL_DIR / "denormalization_vec_B.csv",
}

print("MODEL_DIR:", MODEL_DIR)
print("NEURAL_C:", NEURAL_C)
print("NEURAL_H:", NEURAL_H)
for k, f in files.items():
    print(k, f.exists(), f)


def load_csv_vector(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing CSV: {path}")
    vals = [float(x) for row in csv.reader(path.open()) for x in row if x.strip()]
    if not vals:
        raise ValueError(f"No numeric values in {path}")
    return vals


if APPLY_NORMALIZATION_PATCH:
    c_text = NEURAL_C.read_text()
    for key, fp in files.items():
        vals = load_csv_vector(fp)
        body = ", ".join(f"{v:.9g}f" for v in vals)

        # Match real definitions only (start-of-line float, optional qualifiers, [] or [N])
        pattern = rf"(?ms)^\s*(?:static\s+)?(?:const\s+)?float\s+{re.escape(key)}\s*\[[^\]]*\]\s*=\s*\{{.*?\}}\s*;"
        repl = f"float {key}[] = {{{body}}};"
        c_text, n = re.subn(pattern, repl, c_text, count=1)
        if n != 1:
            raise RuntimeError(f"Could not patch {key} in {NEURAL_C}")

    NEURAL_C.write_text(c_text)
    print("Updated", NEURAL_C)

if UPDATE_NETWORK_HEADER:
    h_text = NEURAL_H.read_text()
    in_n = len(load_csv_vector(files["hls_normalize_a"]))
    out_n = len(load_csv_vector(files["hls_denormalize_A"]))

    h_text, n1 = re.subn(r"(?m)^\s*#define\s+MLP_ACTIVATION_NEURONS\s+\d+", f"#define MLP_ACTIVATION_NEURONS		{in_n}", h_text, count=1)
    h_text, n2 = re.subn(r"(?m)^\s*#define\s+MLP_PREDICTION_NEURONS\s+\d+", f"#define MLP_PREDICTION_NEURONS		{out_n}", h_text, count=1)
    if n1 != 1 or n2 != 1:
        raise RuntimeError(f"Could not patch neuron count defines in {NEURAL_H}")

    NEURAL_H.write_text(h_text)
    print("Updated", NEURAL_H)

if not APPLY_NORMALIZATION_PATCH and not UPDATE_NETWORK_HEADER:
    print("Dry run only. Set APPLY_NORMALIZATION_PATCH and/or UPDATE_NETWORK_HEADER to True.")


### 6.2 Generating the FPGA Bitstream

This cell executes the README Step 6.2 commands directly from Python:
1. `cd ~/physical-cartpole && ./install_zybo_board.sh`
2. `cd ~/physical-cartpole/FPGA/VivadoProjects && vivado -mode batch -source ~/physical-cartpole/generate_bitstream.tcl`
3. Retry with `MALLOC_CHECK_`, `MALLOC_ARENA_MAX`, and `LD_PRELOAD` if the first Vivado run fails.


In [ ]:
IMPL_TMP = IMPL_REPO / ".notebook_impl"
IMPL_TMP.mkdir(exist_ok=True)

def _resolve_tcl_for_current_repo(src_tcl: Path) -> Path:
    """Use repo script as-is when path matches ~/physical-cartpole, otherwise make a patched copy."""
    expected_repo = (Path.home() / "physical-cartpole").resolve()
    if IMPL_REPO.resolve() == expected_repo:
        print("Using original TCL:", src_tcl)
        return src_tcl

    text = src_tcl.read_text()
    repo_str = str(IMPL_REPO)
    text = text.replace("$::env(HOME)/physical-cartpole", repo_str)
    text = text.replace("~/physical-cartpole", repo_str)

    out_tcl = IMPL_TMP / f"{src_tcl.stem}.notebook.tcl"
    out_tcl.write_text(text)
    print("Using notebook-local TCL copy:", out_tcl)
    return out_tcl

def _run_cmd_live(cmd, cwd: Path, env=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    print("cwd:", cwd)
    subprocess.run(cmd, cwd=str(cwd), env=env, check=True)

# README: chmod +x scripts
for rel in ["install_zybo_board.sh", "generate_bitstream.tcl", "generate_vitis_project.tcl"]:
    path = IMPL_REPO / rel
    path.chmod(path.stat().st_mode | 0o111)
    print("Executable bit set:", path)

bitstream_tcl = _resolve_tcl_for_current_repo(IMPL_REPO / "generate_bitstream.tcl")
vivado_projects_dir = IMPL_REPO / "FPGA" / "VivadoProjects"

# README Step 6.2 command 1
_run_cmd_live(["bash", IMPL_REPO / "install_zybo_board.sh"], cwd=IMPL_REPO)

# README Step 6.2 command 2 (+ fallback command with jemalloc env)
vivado = shutil.which("vivado")
if not vivado:
    raise RuntimeError("vivado not found in PATH")

vivado_cmd = [vivado, "-mode", "batch", "-source", bitstream_tcl]
try:
    _run_cmd_live(vivado_cmd, cwd=vivado_projects_dir)
except subprocess.CalledProcessError:
    env = os.environ.copy()
    env["MALLOC_CHECK_"] = "0"
    env["MALLOC_ARENA_MAX"] = "2"
    jemalloc = Path("/usr/lib/x86_64-linux-gnu/libjemalloc.so.2")
    if jemalloc.exists():
        env["LD_PRELOAD"] = str(jemalloc)
    print("First Vivado run failed. Retrying with MALLOC/LD_PRELOAD settings...")
    _run_cmd_live(vivado_cmd, cwd=vivado_projects_dir, env=env)

xsa = IMPL_REPO / "FPGA" / "VivadoProjects" / "CartpoleDriverZynq" / "cartpole_driver_design_wrapper.xsa"
print("XSA exists:", xsa.exists(), xsa)


### 6.3 Generating the SoC Project and `BOOT.bin`

This cell executes the README Step 6.3 command from Python:
`cd ~/physical-cartpole && xsct generate_vitis_project.tcl > vitis_output.log 2>&1`


In [ ]:
vitis_tcl = _resolve_tcl_for_current_repo(IMPL_REPO / "generate_vitis_project.tcl")
xsct = shutil.which("xsct")
if not xsct:
    raise RuntimeError("xsct not found in PATH")

# README Step 6.3 output location
vitis_log = IMPL_REPO / "vitis_output.log"

print("$", f"xsct {vitis_tcl} > {vitis_log} 2>&1")
print("cwd:", IMPL_REPO)
with vitis_log.open("w") as logf:
    subprocess.run(
        [xsct, str(vitis_tcl)],
        cwd=str(IMPL_REPO),
        stdout=logf,
        stderr=subprocess.STDOUT,
        check=True,
    )

print("Vitis log written to:", vitis_log)

boot_bins = sorted((IMPL_REPO / "Firmware" / "VitisProjects").glob("**/BOOT.bin"))
if not boot_bins:
    print("No BOOT.bin found yet. Check:", vitis_log)
else:
    print("Generated BOOT.bin files:")
    for p in boot_bins:
        print(" -", p)

Notes:
- The cells above run the existing automation scripts directly via Python `subprocess`.
- If this repo is not located at `~/physical-cartpole`, the notebook automatically creates temporary TCL copies with corrected absolute paths.
- If Vivado/XSCT are not found, source your Xilinx setup script first (for example `setup_xilinx.sh`) and rerun Step 6.2/6.3 cells.


## Step 7
Load Image on SD card and onto FPGA

This is the final step.

Once the image is successfully loaded onto the SD card and FPGA, the system will be fully configured and ready for operation. There are four switches on the board: the two in the middle serve important functions. One of the switches calibrates the center of the track, while the other allows the cartpole to stabilize either in the upward or downward position. Play with them to see what happens!

Congratulations on completing the setup!


In [ ]:
boot_bins = sorted((IMPL_REPO / "Firmware" / "VitisProjects").glob("**/BOOT.bin"))
print("Found BOOT.bin files:", len(boot_bins))
for p in boot_bins:
    print(" -", p)

if not boot_bins:
    print("No BOOT.bin found. Re-run Step 6.3 first.")
else:
    print("Next (manual/local):")
    print("1) Copy BOOT.bin to SD card root partition")
    print("2) Set board boot mode to SD")
    print("3) Power-cycle board and verify behavior with switches")
